# Hackathon — iNDI ROI Generation

## Background

One endpoint of the iNDI project is organelle morphometrics at single-cell resolution. To assign organelles to a cell, we first define the cell body as a circle (120 px radius, ≈ 11.3 µm) around the nucleus centroid. We then perform organelle segmentation on each channel within that ROI, assigning each organelle to a specific nucleus/cell body.

We use circular ROIs because the panel has **no membrane or cytoplasm marker** — there's no direct signal for the true cell boundary, so the cell body has to be approximated from the nucleus. The fixed-radius circle is a deliberately simple stand-in.

Using this method, we identified ~2,000,000 cell bodies/ROIs across our first antibody panel. In dense regions, neighboring circles overlap, and an organelle in the overlap zone can't be confidently assigned to one cell. We currently drop any ROI whose area overlaps a neighbor by more than 5%, removing ~600,000 (30%) of ROIs.

## The problem

This filtering keeps organelle-to-cell body assignments confident, but it's conservative by design: a fixed threshold that removes ROIs entirely rather than salvaging them. It also points at the deeper objective — what we ultimately care about is correct organelle-to-cell assignment, not ROI count, particularly in crowded regions where a single circle is a rough fit.

## Goal

Develop better approaches to ROI generation or refinement that improve organelle assignment in high-density areas — whether by reshaping circular ROIs, replacing them with a different geometry, or handling overlap zones more intelligently than discard-everything.

Some directions to consider:

- **Bisecting lines** — connect ROI intersection points with a straight line and use it as the division between neighbors
- **Tangent merging** — connect lines between nuclei, find middle tangent lines, merge with the circular ROIs
- **Alternative boundary signal** — use other channels to find the cell boundary (brightfield? over-exposed channels?)
- **Region growing** — bubble formation / mask growing / watershed
- **Shrink-to-fit** — mask erosion
- **Voronoi tessellation** - intersect with circle ROI
- **Cellpose** — learned cell segmentation

## In this notebook

- Load and explore a subset of the iNDI dataset
- Preform the nucleus segmentation
- ROI generation and the current overlap filter
- Visualize where the circle model fails in dense regions
- Examine potential solutions to the problem

In [ ]:
import re
import random
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from scipy.stats import norm
from scipy.optimize import brentq
from scipy.spatial import cKDTree

import tifffile
from skimage import filters, morphology, measure

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle

# View some images

I pulled 50 parental images of nuclei from one plate (Plate INDI00043D; Experiment ID: 4405a3b2-6b88-49b1-91f3-992e09ccbd16). These should have downloaded into a data folder in your directory - reach out if they did not!

These images were aquired on an Opera Phenix and have associated metadata in their folder. Lets take a look at several of these. 

## Set up pseudocoloring - optional

In [ ]:
# Define pseudocolor mapping rules - helpful to generate images on the fly
pseudocolor_map = {
    "DAPI": "blue",
    "Brightfield": "gray",
    "Alexa 488": "green",
    "Alexa 568": "red",
    "Alexa 647": "magenta"
}

# Define pseudocolor mapping rules for matplotlib - helpful to generate images on the fly
mpl_colormaps = {
    "blue": LinearSegmentedColormap.from_list("black_blue", [(0, 0, 0), (0, 0, 1)]),
    "green": LinearSegmentedColormap.from_list("black_green", [(0, 0, 0), (0, 1, 0)]),
    "red": LinearSegmentedColormap.from_list("black_red", [(0, 0, 0), (1, 0, 0)]),
    "magenta": LinearSegmentedColormap.from_list("black_magenta", [(0, 0, 0), (1, 0, 1)]),
    "gray": LinearSegmentedColormap.from_list("black_gray", [(0, 0, 0), (1, 1, 1)])
}

## Access image metadata

In [ ]:
experiment_name = "4405a3b2-6b88-49b1-91f3-992e09ccbd16"

base_path = Path("./data/")
experiment_path = base_path / experiment_name

experiment_xml_file = next(experiment_path.glob("*.xml"), None)

experiment_img_dir = experiment_path / "images"

index_xml = next((experiment_path / "index").glob("*.xml"), None)

tree = ET.parse(experiment_xml_file)
OF_dir = Path(experiment_img_dir)
index_tree = ET.parse(index_xml)

root = tree.getroot()

# Namespace mapping
ns = {'h': '43B2A954-E3C3-47E1-B392-6635266B0DD3/HarmonyV7'} # I believe this is consistent between all experiments

# Find elements
measurement_id = root.find('h:MeasurementID', ns).text
date = root.find('h:Date', ns).text
plate = root.find('h:InitialPlateName', ns).text

print("Measurement ID:", measurement_id)
print("Date:", date)
print("Plate:", plate)

root = index_tree.getroot()

# Find elements
plate_id = root.find('.//h:PlateID', ns).text
x_res = float(root.find('.//h:ImageResolutionX', ns).text) * 1e6
y_res = float(root.find('.//h:ImageResolutionY', ns).text) * 1e6

print("Plate:", plate_id)
print(f"X resolution: {x_res} µm")
print(f"Y resolution: {y_res} µm")

channels = []

# Find the Map element that contains channel info entries
for map_el in root.findall(".//h:Map", ns):
    # Check first entry if it has a ChannelName child
    first_entry = map_el.find("h:Entry", ns)
    if first_entry is not None and first_entry.find("h:ChannelName", ns) is not None:
        # Found the Map with channel entries
        for entry in map_el.findall("h:Entry", ns):
            ch_id = entry.attrib.get("ChannelID")
            ch_name = entry.find("h:ChannelName", ns).text
            ch_type = entry.find("h:ChannelType", ns).text if entry.find("h:ChannelType", ns) is not None else None
            excitation = entry.find("h:MainExcitationWavelength", ns).text if entry.find("h:MainExcitationWavelength", ns) is not None else None
            emission = entry.find("h:MainEmissionWavelength", ns).text if entry.find("h:MainEmissionWavelength", ns) is not None else None

            channels.append({
                "ChannelID": int(ch_id) if ch_id is not None else None,
                "Channel_name": ch_name,
                "Type": ch_type,
                "Excitation_nm": excitation,
                "Emission_nm": emission
            })
        break  

# Convert to DataFrame
channel_df = pd.DataFrame(channels).sort_values('ChannelID').reset_index(drop=True)
print(channel_df)

channel_df["Pseudocolor"] = channel_df["Channel_name"].map(pseudocolor_map).fillna("gray")
channel_df["MPL_colormap"] = channel_df["Pseudocolor"].str.lower().map(mpl_colormaps)
channel_df["Measurement_ID"] = measurement_id
channel_df["Measurement_date"] = date
channel_df["Plate_ID"] = plate_id
channel_df["res_x"] = x_res
channel_df["res_y"] = y_res

# Collect all .tiff files
files = sorted([f for f in OF_dir.rglob('*') if f.suffix.lower() in ['.tiff']])

# Define a function to parse the filename
def parse_filename(name):
    match = re.match(r"r(\d+)c(\d+)f(\d+)p(\d+)-ch(\d+)t(\d+)", name)
    if match:
        return [int(g) for g in match.groups()]
    else:
        return [None] * 6  # Fallback if filename doesn't match

# Create a DataFrame with full path and relative subdirectory
df = pd.DataFrame({
    'filepath': files,
    'filename': [f.name for f in files], 
    'subdirectory': [f.parent.relative_to(OF_dir) for f in files]
})

# Extract metadata from filenames
df[['Row', 'Column', 'Frame', 'Plane', 'ChannelID', 'Time']] = df['filename'].apply(
    lambda x: pd.Series(parse_filename(x))
)

merged_ChannelID_df = pd.merge(df, channel_df, on="ChannelID")

summary = {
    "wells": merged_ChannelID_df[['Row', 'Column']].drop_duplicates().shape[0],
    "channels": merged_ChannelID_df['ChannelID'].nunique(),
    "z_planes": merged_ChannelID_df['Plane'].nunique(),
    "frames": merged_ChannelID_df['Frame'].nunique(),
    "timepoints": merged_ChannelID_df['Time'].nunique(),
}

print(summary)

print(f"""
Experiment ID: {measurement_id}
Plate ID: {plate}
Wells imaged: {summary["wells"]}
Frames per well: {summary["frames"]}
Channels per image: {summary["channels"]}
Z-slices per image: {summary["z_planes"]}
Timepoints per image: {summary["timepoints"]}
""")

## View some random images

In [ ]:
nuc_files = sorted(experiment_img_dir.rglob("*-ch01t01.tiff")) + sorted(experiment_img_dir.rglob("*-ch01t01.tif"))

# Pick 10 at random
random.seed(42) 
sample = random.sample(nuc_files, 10)

# Plot iamges
fig, axes = plt.subplots(2, 5, figsize=(18, 7.5))

for ax, fpath in zip(axes.flat, sample):
    img = tifffile.imread(fpath)

    # Percentile contrast stretch
    lo, hi = np.percentile(img, (0.2, 99.8))
    ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)

    # Label with the well ID
    well = fpath.name.split("f")[0]
    ax.set_title(well, fontsize=10)
    ax.axis("off")

fig.suptitle("Random nucleus images", fontsize=14)
plt.tight_layout()
plt.show()

# Segment nuclei
There are many approaches to nucleus segmentation. Here, I adapted thresholding based approached defined by Allen Cell Segmengeter (https://www.allencell.org/segmenter.html). This is an excellent resource for classical approaches to segmentation! 

In [ ]:
def segment_nucleus(nuc,
                    intensity_scaling_param=(1, 7),
                    blur_sigma=1,
                    min_area=3500):
    # Normalize image
    m, s = norm.fit(nuc.flatten())
    stretch_min = max(m - intensity_scaling_param[0] * s, nuc.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, nuc.max())
    nuc_stretch = np.clip(nuc, stretch_min, stretch_max)
    image_norm = (nuc_stretch - stretch_min) / (stretch_max - stretch_min)

    # Blur
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # Step 1: low-level threshold
    triangle_cutoff = filters.threshold_triangle(blurred)
    global_median_cutoff = np.percentile(blurred, 50)
    th_low_cutoff = (triangle_cutoff + global_median_cutoff) / 2
    img_low_level = blurred > th_low_cutoff
    img_low_level = morphology.remove_small_objects(
        img_low_level, min_size=min_area, connectivity=1, out=img_low_level)
    img_low_level = morphology.dilation(img_low_level, footprint=morphology.disk(2))

    # Step 2: high-level threshold
    otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
    img_high_level = np.zeros_like(img_low_level)
    lab_low, num_obj = morphology.label(img_low_level, return_num=True, connectivity=1)
    for idx in range(num_obj):
        single_obj = lab_low == (idx + 1)
        local_otsu = filters.threshold_otsu(blurred[single_obj])
        if local_otsu > otsu_cutoff:
            img_high_level[np.logical_and(blurred > 0.98 * local_otsu, single_obj)] = 1

    filled = ndi.binary_fill_holes(img_high_level)
    filled = morphology.dilation(filled, footprint=morphology.disk(2))
    labeled_filled = morphology.label(filled, connectivity=1)
    nuc_seg = morphology.remove_small_objects(labeled_filled, min_size=min_area)

    return nuc_seg

# Pick 5 random nucleus images
sample = random.sample(nuc_files, 5)

# Segment + plot raw vs segmented
fig, axs = plt.subplots(5, 2, figsize=(10, 25))
fig.suptitle("Nucleus segmentation: raw vs. final mask", fontsize=14, y=0.995)

for row, fpath in enumerate(sample):
    nuc = tifffile.imread(fpath)
    nuc_seg = segment_nucleus(nuc)

    well = fpath.name.split("f")[0]

    lo, hi = np.percentile(nuc, (0.2, 99.8))
    axs[row, 0].imshow(nuc, cmap="gray", vmin=lo, vmax=hi)
    axs[row, 0].set_title(f"{well} — raw (DAPI)")
    axs[row, 0].axis("off")

    axs[row, 1].imshow(nuc_seg)
    n_nuclei = len(np.unique(nuc_seg)) - 1  # minus background
    axs[row, 1].set_title(f"{well} — segmented ({n_nuclei} nuclei)")
    axs[row, 1].axis("off")

plt.tight_layout()
plt.show()

# Generate ROIs
We generate cell body ROIs by drawing a circle around the nucleus centroid that has a radius of 120 pixels. Obviously, it is not uncommon that there are overlapping ROIs. 

In [ ]:
# Define ROI radius
RADIUS = 120

# Lazy 5% overlap measurement
# Find the center-distance threshold for >=5% area overlap
def overlap_area(d, r):
    if d >= 2 * r:
        return 0.0
    if d <= 0:
        return np.pi * r**2
    return (2 * r**2 * np.arccos(d / (2 * r))
            - (d / 2) * np.sqrt(4 * r**2 - d**2))

target = 0.05 * np.pi * RADIUS**2
# Centroid distance at which overlap area == 5% of circle area
dist_threshold = brentq(lambda d: overlap_area(d, RADIUS) - target, 1e-6, 2 * RADIUS)

# Per-image: centroids -> circles -> overlap count
def analyze_image(nuc_seg):
    props = measure.regionprops(nuc_seg)
    centroids = np.array([p.centroid for p in props])  # (row, col) = (y, x)
    if len(centroids) < 2:
        return centroids, np.zeros(len(centroids), dtype=bool)

    tree = cKDTree(centroids)
    # Pairs of centroids closer than the threshold => >=5% overlap
    pairs = tree.query_pairs(r=dist_threshold)
    overlapping = np.zeros(len(centroids), dtype=bool)
    for i, j in pairs:
        overlapping[i] = True
        overlapping[j] = True
    return centroids, overlapping

# View 5 random images
sample = random.sample(nuc_files, 5)

fig, axs = plt.subplots(1, 5, figsize=(25, 5.5))
fig.suptitle(f"{RADIUS}px circles on nuclei centroids — overlapping circles in red", fontsize=13)

for ax, fpath in zip(axs, sample):
    nuc = tifffile.imread(fpath)
    nuc_seg = segment_nucleus(nuc)
    centroids, overlapping = analyze_image(nuc_seg)

    lo, hi = np.percentile(nuc, (0.2, 99.8))
    ax.imshow(nuc, cmap="gray", vmin=lo, vmax=hi)

    for (y, x), is_ov in zip(centroids, overlapping):
        color = "red" if is_ov else "lime"
        ax.add_patch(Circle((x, y), RADIUS, fill=False, edgecolor=color, linewidth=1.2))

    well = fpath.name.split("f")[0]
    n_total = len(centroids)
    n_ov = int(overlapping.sum())
    ax.set_title(f"{well}\n{n_ov}/{n_total} circles overlap (\u22655%)", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()